In [ ]:
import json
import random

# 1. Train - Dev - Test Splitting

In [ ]:
input_file = "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/raw/output_ner.jsonl"

train_file = "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/train_raw.jsonl"
dev_file = "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/dev_raw.jsonl"
test_file = "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/test_raw.jsonl"

# Đọc dữ liệu
data = []
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))

# Shuffle (xáo trộn)
random.shuffle(data)

# Tính số lượng cho mỗi tập
n = len(data)
n_train = int(n * 0.7)
n_dev = int(n * 0.15)
n_test = n - n_train - n_dev   # phần còn lại

train_data = data[:n_train]
dev_data = data[n_train:n_train + n_dev]
test_data = data[n_train + n_dev:]

# Hàm ghi file jsonl
def write_jsonl(filename, rows):
    with open(filename, "w", encoding="utf-8") as f:
        for obj in rows:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

write_jsonl(train_file, train_data)
write_jsonl(dev_file, dev_data)
write_jsonl(test_file, test_data)

print("Done!")
print("Train:", len(train_data))
print("Dev:", len(dev_data))
print("Test:", len(test_data))

Done!
Train: 41422
Dev: 8876
Test: 8877


# 2. Remove ID

In [ ]:
files = [
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/train_raw.jsonl", 
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/dev_raw.jsonl", 
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/test_raw.jsonl"
]

for filename in files:
    print("Đang xử lý:", filename)

    new_rows = []
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            obj.pop("id", None)  # XÓA ID
            new_rows.append(obj)

    # Ghi lại file (không có id)
    with open(filename, "w", encoding="utf-8") as f:
        for obj in new_rows:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("🎉 Đã xoá xong trường id trong train/dev/test!")


Đang xử lý: train.jsonl
Đang xử lý: dev.jsonl
Đang xử lý: test.jsonl
🎉 Đã xoá xong trường id trong train/dev/test!


# 3. Check invalid rows

- Tags / Tokens rỗng

- File jsonl định dạng không hợp lệ

- Không có Tags hay Tokens

In [5]:
files = [
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/train_raw.jsonl", 
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/dev_raw.jsonl", 
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/test_raw.jsonl"
]

for filename in files:
    print("\n=== Kiểm tra file:", filename, "===")
    error_lines = []

    with open(filename, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
            except:
                error_lines.append((idx, "Không parse được JSON", line))
                continue

            keys = list(obj.keys())

            # dòng lỗi nếu:
            # - không có tokens
            # - không có tags
            # - hoặc key rỗng
            if ("tokens" not in obj) or ("tags" not in obj) or ("" in obj) or (len(keys) == 0):
                error_lines.append((idx, keys, line))

    if not error_lines:
        print("✔ Không có dòng lỗi!")
    else:
        print(f"❌ Tổng số dòng lỗi: {len(error_lines)}")
        for err in error_lines[:20]:  # in tối đa 20 dòng đầu cho dễ xem
            print("\n--- Lỗi ---")
            print("Dòng:", err[0])
            print("Keys:", err[1])
            print("Nội dung:", err[2])

        if len(error_lines) > 20:
            print(f"... còn {len(error_lines) - 20} lỗi nữa ...")


=== Kiểm tra file: /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/train_raw.jsonl ===
✔ Không có dòng lỗi!

=== Kiểm tra file: /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/dev_raw.jsonl ===
✔ Không có dòng lỗi!

=== Kiểm tra file: /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/test_raw.jsonl ===
✔ Không có dòng lỗi!


# 4. Remove samples with NULL keys

In [ ]:
files = [
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/train_raw.jsonl", 
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/dev_raw.jsonl", 
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/test_raw.jsonl"
]
cleaned = []

for filename in files:
    print("Đang xử lý:", filename)
    cleaned = []
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            obj = json.loads(line)

            # Nếu dòng có key rỗng "" → bỏ luôn, không giữ
            if "" in obj:
                continue

            cleaned.append(obj)

    # Ghi đè file train.jsonl sạch
    with open(filename, "w", encoding="utf-8") as f:
        for obj in cleaned:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")
    file_name = filename.split('/')[-1]
    print(f"🎉 Đã xóa các dòng lỗi chứa key rỗng trong {file_name}!")


🎉 Đã xóa các dòng lỗi chứa key rỗng trong train.jsonl!


# 5. Remove Invalid Keys

- Xoá keys "sentence" không hợp lệ trong quá trình gán nhãn

In [ ]:
files = [
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/train_raw.jsonl", 
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/dev_raw.jsonl", 
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/test_raw.jsonl"
]

for filename in files:
    print("Đang xử lý:", filename)
    cleaned = []

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            obj = json.loads(line)

            # XÓA KEY "sentence" nếu tồn tại
            if "sentence" in obj:
                obj.pop("sentence")

            cleaned.append(obj)

    # Ghi đè lại file
    with open(filename, "w", encoding="utf-8") as f:
        for obj in cleaned:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("🎉 Đã xoá 'sentence' trong 3 file train, dev, test!")

Đang xử lý: train.jsonl
Đang xử lý: dev.jsonl
Đang xử lý: test.jsonl
🎉 Đã xoá 'sentence' trong 3 file train, dev, test!


# 6. Remove rows without tokens or tags

In [ ]:
import json

files = ["train.jsonl", "dev.jsonl", "test.jsonl"]

for filename in files:
    print(f"\n===== Xử lý file: {filename} =====")

    good_lines = []
    removed = 0

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
            except:
                removed += 1
                continue

            tokens = obj.get("tokens")
            tags = obj.get("tags")

            # Bỏ dòng không có tokens hoặc tags
            if tokens is None or tags is None:
                removed += 1
                continue

            if len(tokens) != len(tags):
                removed += 1
                continue

            good_lines.append(line)

    # Ghi lại file đã làm sạch
    with open(filename, "w", encoding="utf-8") as out:
        for gl in good_lines:
            out.write(gl + "\n")

    print(f"✔ Giữ lại {len(good_lines)} dòng")
    print(f"❌ Đã xoá {removed} dòng mismatch")



===== Xử lý file: train.jsonl =====
✔ Giữ lại 36751 dòng
❌ Đã xoá 4669 dòng mismatch

===== Xử lý file: dev.jsonl =====
✔ Giữ lại 7890 dòng
❌ Đã xoá 986 dòng mismatch

===== Xử lý file: test.jsonl =====
✔ Giữ lại 7852 dòng
❌ Đã xoá 1025 dòng mismatch


SỬA DỮ LIỆU

In [ ]:
import json

file_path = "test.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    a = []
    for idx, line in enumerate(f):
        data = json.loads(line)
        tags = data["tags"]

        # Chỉ cần có ORG là in index
        # CHỈ đúng khi có phần tử == "ORG"
        if "B" in tags:
            a.append(idx+1)
    print(a)

[]
